In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import time
import transformers
import torch
import torch.nn as nn
import ast
import os
from tqdm import tqdm
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, utils, BertModel

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
df_test = pd.read_csv('path/to/test/split.csv') # < --- Update the path to your test split CSV file
df_val = pd.read_csv('path/to/val/split.csv') # < --- Update the path to your validation split CSV file
df_train_resampled = pd.read_csv('path/to/train/split.csv') # < --- Update the path to your training split CSV file

In [ ]:
list_of_strings_train=df_train_resampled['input_ids'].tolist()
list_of_strings_train = [str(item) for item in list_of_strings_train] 

list_of_strings_val=df_val['input_ids'].tolist()
list_of_strings_val = [str(item) for item in list_of_strings_val]

list_of_strings_test=df_test['input_ids'].tolist()
list_of_strings_test = [str(item) for item in list_of_strings_test]



list_of_strings_train_labels=df_train_resampled['label'].tolist()
list_of_strings_val_labels=df_val['label'].tolist()
list_of_strings_test_labels=df_test['label'].tolist()

print(len(list_of_strings_train))
print(len(list_of_strings_val))


print(len(list_of_strings_train_labels))
print(len(list_of_strings_val_labels))

In [ ]:
#now we tokenise (use med-bert??) for TRAIN data 
from transformers import AutoTokenizer, AutoModel, BertTokenizer, BertModel
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")


train=tokenizer(list_of_strings_train, return_tensors='pt', max_length=512, truncation=True, padding='max_length')
train['label']=list_of_strings_train_labels

torch.save(train, 'path/to/train_tokenised.pth') # < --- Update the path where you want to save the tokenised training data

val=tokenizer(list_of_strings_val, return_tensors='pt', max_length=512, truncation=True, padding='max_length')
val['label']=list_of_strings_val_labels

torch.save(val, 'path/to/val_tokenised.pth') # < --- Update the path where you want to save the tokenised validation data


test=tokenizer(list_of_strings_test, return_tensors='pt', max_length=512, truncation=True, padding='max_length')
test['label']=list_of_strings_test_labels

torch.save(test, 'path/to/test_tokenised.pth') # < --- Update the path where you want to save the tokenised test data



In [ ]:
from torch.utils.data import Dataset, DataLoader, TensorDataset

class CustomDataset(Dataset):
    def __init__(self, encodings): 
        self.encodings=encodings
    def __len__(self):
        return len(self.encodings['input_ids'])
    
    def __getitem__(self, idx):     

        sample = {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.encodings['label'][idx]
        }
        return sample
    
train = torch.load('path/to/train_tokenised.pth', weights_only = False) # < --- Update the path to where you saved the tokenised training data
val = torch.load('path/to/val_tokenised.pth', weights_only = False) # < --- Update the path to where you saved the tokenised validation data

batch_size = 16 

train_dataset = CustomDataset(train)
train_data_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_dataset = CustomDataset(val)
val_data_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "allenai/biomed_roberta_base" 
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, weights_only=False)

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU.")
else:
    print("No GPU available, using the CPU instead.")
    device = torch.device("cpu")

model.to(device)

class_weights = torch.tensor([1.0, 3.0]).to(device) 

loss_function = nn.CrossEntropyLoss(weight=class_weights)


In [ ]:
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

optimizer = AdamW(model.parameters(),
                  lr=2e-6,
                  eps=1e-8)

epochs = 25
total_steps = len(train_data_loader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer,       
                 num_warmup_steps=0, num_training_steps=total_steps)


In [ ]:
import csv
import torch
from torch.nn.utils.clip_grad import clip_grad_norm_
from tqdm.notebook import tqdm
import wandb

def train(
    model,
    optimizer,
    scheduler,
    epochs,
    loss_function,
    train_data_loader,
    val_data_loader,
    device,
    clip_value=2.0,
    csv_file=None,
    # early stopping
    patience=3,
    min_delta=0.0,
    # wandb
    use_wandb=True,
    wandb_project="Project-Name", # < --- Update with your W&B project name
    wandb_run_name=None,
    wandb_config=None,
    log_every_steps=200,
    best_model_path="path/to/best_model.pth", # < --- Update with the path where you want to save the best model
):
    """
    Early stopping criteria:
      - If (best_val_loss - avg_val_loss) <= min_delta, counts as no improvement.
      - Stop when no_improve_count >= patience.
    """

    # --- W&B init ---
    run = None
    if use_wandb:
        run = wandb.init(
            project=wandb_project,
            name=wandb_run_name,
            config=wandb_config or {},
            reinit=True,
        )

    best_val_loss = float("inf")
    no_improve_count = 0

    train_loss_per_epoch = []
    val_loss_per_epoch = []

    csv_writer = None
    if csv_file:
        csv_writer = csv.writer(csv_file)
        csv_writer.writerow(["Epoch", "Train Loss", "Val Loss"])

    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}/{epochs}")
        print("Training...")

        model.train()
        total_loss = 0.0

        for step, batch in enumerate(tqdm(train_data_loader, desc="Training")):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad(set_to_none=True)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            loss = loss_function(logits, labels)

            loss.backward()
            grad_norm = clip_grad_norm_(model.parameters(), clip_value)

            optimizer.step()
            scheduler.step()

            total_loss += loss.item()

            # per-step logging
            if (step + 1) % log_every_steps == 0:
                lr = optimizer.param_groups[0]["lr"]
                print(f"Step {step + 1}/{len(train_data_loader)}, Loss: {loss.item():.6f}, LR: {lr:.3e}")

                if use_wandb:
                    wandb.log(
                        {
                            "train/loss_step": loss.item(),
                            "train/grad_norm": float(grad_norm) if grad_norm is not None else None,
                            "train/lr": lr,
                            "epoch": epoch + 1,
                            "step": epoch * len(train_data_loader) + (step + 1),
                        }
                    )

        avg_train_loss = total_loss / max(1, len(train_data_loader))
        train_loss_per_epoch.append(avg_train_loss)
        print(f"Average training loss: {avg_train_loss:.6f}")

        print("Validation...")
        model.eval()
        total_val_loss = 0.0

        with torch.no_grad():
            for val_step, val_batch in enumerate(tqdm(val_data_loader, desc="Validation")):
                val_input_ids = val_batch["input_ids"].to(device)
                val_attention_mask = val_batch["attention_mask"].to(device)
                val_labels = val_batch["labels"].to(device)

                val_outputs = model(input_ids=val_input_ids, attention_mask=val_attention_mask)
                val_logits = val_outputs.logits
                val_loss = loss_function(val_logits, val_labels)

                total_val_loss += val_loss.item()

        avg_val_loss = total_val_loss / max(1, len(val_data_loader))
        val_loss_per_epoch.append(avg_val_loss)
        print(f"Average validation loss: {avg_val_loss:.6f}")

        # epoch-level wandb logging
        if use_wandb:
            wandb.log(
                {
                    "train/loss_epoch": avg_train_loss,
                    "val/loss": avg_val_loss,
                    "epoch": epoch + 1,
                }
            )

        # Save best model + early stopping bookkeeping
        improved = (best_val_loss - avg_val_loss) > min_delta
        if improved:
            best_val_loss = avg_val_loss
            no_improve_count = 0
            torch.save(model.state_dict(), best_model_path)
            print(f"Model improved; saved to {best_model_path}")

            if use_wandb:
                import wandb
                wandb.run.summary["best_val_loss"] = best_val_loss
                wandb.run.summary["best_epoch"] = epoch + 1
                
        else:
            no_improve_count += 1
            print(f"No improvement for {no_improve_count}/{patience} epoch(s).")

        if csv_writer:
            csv_writer.writerow([epoch + 1, avg_train_loss, avg_val_loss])

        # EARLY STOP
        if no_improve_count >= patience:
            print(f"Early stopping: no val-loss improvement for {patience} epochs.")
            break

    if use_wandb and run is not None:
        run.finish()

    return model


csv_file_path = "path/to/training_results.csv"
with open(csv_file_path, "w", newline="") as csvfile:
    model = train(
        model, optimizer, scheduler, epochs, loss_function,
        train_data_loader, val_data_loader, device,
        clip_value=2.0,
        csv_file=csvfile,
        patience=5,           # x epochs
        min_delta=1e-4,        # optional threshold
        use_wandb=True,
        wandb_project="Project-Name", # < --- Update with your W&B project name
        wandb_run_name="ROBERTA_mimic", # < --- Update with your W&B run name
        wandb_config={
            "epochs": epochs,
            "clip_value": 2.0,
            "patience": 5,
            "min_delta": 1e-4,
        },
    )

In [ ]:
test=torch.load("path/to/test_tokenised.pth") # < --- Update the path to where you saved the tokenised test data

model_name = "allenai/biomed_roberta_base" 
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, weights_only = False)  

model.load_state_dict(torch.load("path/to/best_model.pth")) # < --- Update the path to where you saved the best model during training

if torch.cuda.is_available():       
    device = torch.device("cuda")
    print("Using GPU.")
else:
    print("No GPU available, using the CPU instead.")
    device = torch.device("cpu")

model.to(device)

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, encodings): # Constructor
        self.encodings=encodings #allows us to retrieve the values passed by calling the encodings attribute
    def __len__(self):
        return len(self.encodings['input_ids'])
    
    def __getitem__(self, idx):     # getting the data

        sample = {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.encodings['label'][idx]
        }
        return sample
    
batch_size = 16 

test_dataset = CustomDataset(test)
test_data_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import KFold

num_classes = 2
k_folds = 5

results = []

kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

indices = np.arange(len(test_data_loader.dataset))

for fold, (train_idx, test_idx) in enumerate(kf.split(indices)):
    print(f'Fold {fold+1}/{k_folds}')
    
    train_sampler = torch.utils.data.SubsetRandomSampler(train_idx)
    test_sampler = torch.utils.data.SubsetRandomSampler(test_idx)
    
    train_loader = torch.utils.data.DataLoader(
        test_data_loader.dataset,
        sampler=train_sampler,
        batch_size=test_data_loader.batch_size
    )
    
    test_loader = torch.utils.data.DataLoader(
        test_data_loader.dataset,
        sampler=test_sampler,
        batch_size=test_data_loader.batch_size
    )
    
    model.eval()
    
    all_probabilities = [] 
    all_predictions = []
    all_labels = []
    test_loss = 0
    
    with torch.no_grad():
        for step_num, batch_data in enumerate(test_loader):
            
            input_ids = batch_data['input_ids'].to(device)
            attention_mask = batch_data['attention_mask'].to(device)
            labels = batch_data['labels'].to(device)
            
            output = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            
            logits = output.logits
            loss = loss_function(logits, labels)
            test_loss += loss.item()

            probabilities = F.softmax(logits, dim=-1)

            predictions = torch.argmax(probabilities, dim=1)

            all_probabilities.extend(probabilities.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())
            
    accuracy = accuracy_score(all_labels, all_predictions)
    
    report = classification_report(
        all_labels,
        all_predictions,
        target_names=[f'Class {i}' for i in range(num_classes)],
        output_dict=True
    )
        
    fold_result = {
        'Fold': fold+1, 
        'accuracy': accuracy, 
        'short stay precision': report['Class 0']['precision'],
        'short stay recall': report['Class 0']['recall'],
        'short stay f1': report['Class 0']['f1-score'],
        'long stay precision': report['Class 1']['precision'],
        'long stay recall': report['Class 1']['recall'],
        'long stay f1': report['Class 1']['f1-score'],
        'weighted stay precision': report['weighted avg']['precision'],
        'weighted stay recall': report['weighted avg']['recall'],
        'weighted stay f1': report['weighted avg']['f1-score']
    }
    
    results.append(fold_result)
    
results_df = pd.DataFrame(results)
metrics_df = results_df.drop(columns='Fold').agg(['mean', 'std'])

In [ ]:
results_df

In [ ]:
metrics_df